# 🌴 Tropical Diseases — Knowledge Graph Diagnostic Evaluation (Kaggle GPU Edition)
### Interactive Querying, Multi-Mode Benchmarks, Health Inspection & Graph Explorer

This notebook connects directly to your **pre-built Tropical Diseases Knowledge Graph** (indexed from 1,797 clinical case reports) on **Kaggle GPU** using `tropical_kg_storage.zip`.

**Kaggle Execution Advantages:**
- ⚡ **Dual T4 GPUs (30 GB VRAM total) or P100 (16 GB):** Blazing fast query evaluation with zero memory pressure.
- 🚀 **Zero Information Loss:** Evaluates 100% of the raw Knowledge Graph context in a native 16k GPU context buffer (`num_ctx: 16384`).
- ⏱️ **Fast NVMe Extraction:** Automatically unzips `tropical_kg_storage.zip` into Kaggle working memory in ~5 seconds.
- 🛡️ **Read-Only Consultation Safety:** Queries without mutating the underlying graph database.

| Kaggle Settings Required | Value |
| :--- | :--- |
| **Accelerator** | **GPU T4 x 2** (or GPU P100) |
| **Internet** | **On** (Toggle in right-hand sidebar under Settings) |
| **Attached Dataset** | Upload `tropical_kg_storage.zip` as input dataset |

---
## Step 1: Environment & GPU Configuration (~1 minute) ⚡
1. Installs `pciutils` so Ollama detects the NVIDIA T4 GPU driver.
2. Configures `OLLAMA_MAX_LOADED_MODELS=2` and `OLLAMA_KEEP_ALIVE=24h` so both **`bge-m3`** and **`qwen2.5:3b`** live concurrently in VRAM (taking only ~3.1 GB out of Colab's 15 GB VRAM).
3. Pulls `bge-m3` and `qwen2.5:3b`.

In [ ]:
import os, sys, subprocess, time, urllib.request

print("🌐 Running on Kaggle GPU Environment")
!nvidia-smi

# 1. Install pciutils and dependencies for Ollama on Kaggle
!apt-get update -qq && apt-get install -y -qq pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio networkx matplotlib
print("✅ System dependencies & Ollama installed!")

# 2. Kill any stale Ollama processes from earlier runs
!pkill -9 -f ollama > /dev/null 2>&1 || true
time.sleep(2)

# --- Configure Ollama for High-Speed GPU Concurrency ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OLLAMA_KEEP_ALIVE"] = "24h"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "2"    # Keep both bge-m3 & qwen2.5:3b permanently in VRAM
os.environ["OLLAMA_NUM_PARALLEL"] = "2"

ollama_bin = "ollama"
log_path = "/kaggle/working/ollama.log"

print(f"Starting Ollama server (logging to {log_path})...")
with open(log_path, "w") as log_file:
    subprocess.Popen([ollama_bin, "serve"], stdout=log_file, stderr=log_file, env=os.environ)

time.sleep(4)
print("✅ Ollama server started!")

# --- Models ---
EMBEDDING_MODEL = "bge-m3"
QUERY_LLM_MODEL = "qwen2.5:3b"

print(f"\nPulling / verifying embedding model ({EMBEDDING_MODEL})...")
!ollama pull {EMBEDDING_MODEL}

print(f"\nPulling / verifying query LLM ({QUERY_LLM_MODEL})...")
!ollama pull {QUERY_LLM_MODEL}

print("\n--- Installed Models ---")
!ollama list

---
### Step 1.5: GPU Sanity Check (1 Second) 🔍
Runs a rapid 1-sentence prompt directly through Ollama and verifies that `PROCESSOR` reports **`100% GPU`**.

In [ ]:
# Quick sanity check: verify sub-second GPU inference
import time, ollama

t0 = time.time()
resp = ollama.chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": "Respond with 'GPU acceleration active' in 3 words."}]
)
t_elapsed = time.time() - t0

print(f"✅ Response ({t_elapsed:.2f}s): {resp['message']['content'].strip()}\n")
print("--- GPU Memory Status ---")
!ollama ps

---
## Step 2: Auto-Extract & Connect to Knowledge Graph 📦
Automatically searches for `tropical_kg_storage.zip` in your Kaggle inputs, extracts it to `/kaggle/working`, and verifies graph health:
- Verifies all **1,797 clinical cases** are present.
- Inspects `graph_chunk_entity_relation.graphml` to report total nodes and relationship edges.
- Displays the **Top 15 Hub Entities** in your tropical disease graph.

In [ ]:
import os, sys, json, glob, zipfile, shutil
import networkx as nx

WORKING_DIR = "/kaggle/working"
STORAGE_DIR = os.path.join(WORKING_DIR, "tropical_kg_storage")
WORKSPACE = "tropical_kg"
WORKSPACE_DIR = os.path.join(STORAGE_DIR, WORKSPACE)

print("=" * 65)
print("📊 TROPICAL DISEASES KNOWLEDGE GRAPH KAGGLE SETUP")
print("=" * 65)

# 1. Automatic Zip Discovery & Fast Extraction
if not os.path.exists(WORKSPACE_DIR):
    print("🔍 Searching for tropical_kg_storage.zip or dataset in Kaggle...")
    
    # Check for zip file across /kaggle/input and /kaggle/working
    zip_files = (
        glob.glob("/kaggle/input/**/tropical*storage*.zip", recursive=True) +
        glob.glob("/kaggle/input/**/*.zip", recursive=True) +
        glob.glob("/kaggle/working/**/*.zip", recursive=True)
    )
    
    # Check if Kaggle already auto-unzipped the dataset
    unzipped_matches = glob.glob("/kaggle/input/**/kv_store_full_docs.json", recursive=True)
    
    if zip_files:
        target_zip = zip_files[0]
        print(f"📦 Found archive: {target_zip}")
        print(f"⚡ Extracting to {WORKING_DIR} (takes ~5s on Kaggle NVMe SSD)...")
        with zipfile.ZipFile(target_zip, 'r') as zf:
            zf.extractall(WORKING_DIR)
        print("✅ Extraction complete!")
    elif unzipped_matches:
        source_workspace = os.path.dirname(unzipped_matches[0])
        source_storage = os.path.dirname(source_workspace)
        print(f"📁 Found existing unzipped dataset at: {source_storage}")
        print("Copying to writable working directory...")
        shutil.copytree(source_storage, STORAGE_DIR, dirs_exist_ok=True)
        print("✅ Copied to working storage!")
    else:
        print("⚠️ Could not find tropical_kg_storage.zip automatically.")
        print("Contents of /kaggle/input:")
        !ls -la /kaggle/input/
        raise FileNotFoundError(
            "❌ Could not find tropical_kg_storage.zip in /kaggle/input or /kaggle/working!\n"
            "Please upload tropical_kg_storage.zip as a dataset to your Kaggle notebook."
        )

# 2. Check GraphML file & create symlink if needed (handles .graphml.xml)
graph_file = os.path.join(WORKSPACE_DIR, "graph_chunk_entity_relation.graphml")
xml_file = os.path.join(WORKSPACE_DIR, "graph_chunk_entity_relation.graphml.xml")
if not os.path.exists(graph_file) and os.path.exists(xml_file):
    try:
        os.symlink(xml_file, graph_file)
    except Exception:
        shutil.copyfile(xml_file, graph_file)

# 3. Health Check Diagnostics
doc_file = os.path.join(WORKSPACE_DIR, "kv_store_full_docs.json")
if os.path.exists(doc_file):
    with open(doc_file, "r", encoding="utf-8") as f:
        docs = json.load(f)
    print(f"✅ Indexed Patient Cases: {len(docs)} / 1797 cases")
else:
    print("⚠️ kv_store_full_docs.json not found")

if os.path.exists(graph_file):
    print(f"Loading GraphML ({os.path.getsize(graph_file) / (1024*1024):.2f} MB)...")
    G = nx.read_graphml(graph_file)
    print(f"✅ Total Clinical Entities (Nodes):     {G.number_of_nodes():,}")
    print(f"✅ Total Diagnostic Relations (Edges): {G.number_of_edges():,}")
    print(f"✅ Graph Density:                       {nx.density(G):.6f}")
    print(f"✅ Connected Components:                {nx.number_connected_components(G)}")

    # 4. Top Hub Entities
    print("\n🏆 Top 15 Most Connected Hub Entities in the Knowledge Graph:")
    print("-" * 65)
    sorted_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:15]
    for rank, (node, deg) in enumerate(sorted_nodes, 1):
        node_data = G.nodes[node]
        entity_type = node_data.get("entity_type", "Unknown")
        print(f"  {rank:2d}. [{entity_type:15s}] {node:<35s} (connections: {deg})")
else:
    print("⚠️ graph_chunk_entity_relation.graphml not found")
print("=" * 65)

---
## Step 3: Initialize LightRAG in Query-Only Mode 🚀
Initializes the `LightRAG` client pointing directly to the verified `tropical_kg` workspace.
- **Read-Only Safety:** Protects the pre-built graph from any ingestion mutations.
- **Ollama Integration:** Uses `bge-m3` for vector search and `qwen2.5:3b` for query keywords.

In [ ]:
import asyncio
from functools import partial
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
from lightrag.utils import EmbeddingFunc

try:
    import nest_asyncio
except ImportError:
    import nest_asyncio2 as nest_asyncio
nest_asyncio.apply()

EMBED_DIM = 1024

print("Initializing LightRAG Query Engine...")
rag = LightRAG(
    working_dir=STORAGE_DIR,
    workspace=WORKSPACE,
    llm_model_max_async=2,          # Match Ollama concurrency
    embedding_func_max_async=2,     # Prevent flooding Ollama embedding queue
    llm_model_func=ollama_model_complete,
    llm_model_name=QUERY_LLM_MODEL,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=4096,
        func=partial(ollama_embed.func, embed_model=EMBEDDING_MODEL)
    )
)

await rag.initialize_storages()
print(f"✅ LightRAG successfully initialized in Read-Only Query Mode!")
print(f"   Model: {QUERY_LLM_MODEL} | Embedding: {EMBEDDING_MODEL} (1024 dims)")

---
## Step 4: Two-Stage Clinical Diagnostic Consultation Engine 🔬
*(Direct GPU Synthesis with 100% Full Knowledge Graph Evidence — Zero Dropping)*

1. **Stage 1 (Graph Retrieval ~8–10s):** LightRAG traverses the knowledge graph and retrieves relevant clinical entities, relationships, and patient cases.
2. **Zero Information Loss (0.001s):** Retains 100% of all retrieved case findings, drug resistances, and multi-organ manifestations (no truncation).
3. **Stage 2 (Expert Clinical Consultation on GPU ~15–20s):** Qwen evaluates the full clinical context natively in T4 GPU VRAM (`num_ctx: 16384`) and synthesizes an authoritative, structured diagnostic consultation.

In [ ]:
import time, ollama

async def run_diagnostic_query(
    query: str,
    mode: str = "hybrid",
    chunk_top_k: int = 3,        # Top relevant patient case reports
    top_k: int = 20,             # Top clinical entity/relation nodes
    max_total_tokens: int = 12000 # Generous retrieval token quota
) -> str:
    """
    Two-Stage Zero-Dropping Clinical Consultation Engine:
    Stage 1: Full Knowledge Graph retrieval (~8-10s)
    Formatting: Retains 100% of case findings (<SEP> -> clean semicolons)
    Stage 2: Expert Physician Consultation on T4 Tensor Cores (num_ctx: 16384)
    """
    print(f"🔎 Query: '{query}'")
    print(f"⚙️ Search Mode: [{mode.upper()}] (chunks: {chunk_top_k}, graph nodes: {top_k})")
    
    # --- Stage 1: Knowledge Graph Retrieval ---
    t0 = time.time()
    context_data = await rag.aquery(
        query,
        param=QueryParam(
            mode=mode,
            top_k=top_k,
            chunk_top_k=chunk_top_k,
            max_total_tokens=max_total_tokens,
            only_need_context=True,
            enable_rerank=False
        )
    )
    t_retrieval = time.time() - t0
    
    if not context_data:
        print("⚠️ No relevant context found in Knowledge Graph.")
        return "No relevant graph context found."
    
    # Format <SEP> cleanly without dropping a single character or case finding
    clean_context = context_data.replace("<SEP>", " ; ")
    
    print(f"✅ Graph Retrieved in {t_retrieval:.2f}s (Full Context: {len(clean_context):,} chars | Zero Dropped)")
    
    # --- Stage 2: Expert Clinical Consultation on GPU (16k Native Context) ---
    system_instruction = f"""You are an expert infectious disease specialist and clinical diagnostic consultant.
Provide an exhaustive, clinician-grade differential diagnosis based on the provided Tropical Diseases Knowledge Graph evidence.

Instructions:
1. Ground your response strictly in the provided Knowledge Graph context without hallucinating external facts.
2. Structure your consultation into clear sections:
   - 🩺 Clinical Presentation & Organ Manifestations (highlight both typical and atypical/disseminated findings)
   - 🔬 Laboratory, Microbiological & Molecular Confirmation (specifying culture types, stains, PCR, GeneXpert/CBNAAT)
   - 🩻 Radiologic & Imaging Distinctions (contrasting chest imaging patterns and extrapulmonary findings)
   - 💊 Pharmacotherapy & Drug Resistance Nuances (including resistance mutations like katG/rpoB and standard regimens if present)
   - ⚖️ Key Diagnostic Discriminator Summary
3. Maintain an authoritative, structured, and clinically actionable tone suitable for medical evaluation.

--- Clinical Knowledge Graph Context ---
{clean_context}"""

    t1 = time.time()
    client = ollama.AsyncClient()
    response = await client.chat(
        model=QUERY_LLM_MODEL,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": query}
        ],
        options={
            "temperature": 0.2,
            "num_ctx": 16384  # Native 16k GPU context buffer (576 MB KV cache)
        }
    )
    t_gen = time.time() - t1
    total_time = t_retrieval + t_gen
    
    answer = response["message"]["content"]
    
    print("=" * 70)
    print(f"🏥 Clinical Diagnostic Consultation (Mode: {mode.upper()})")
    print(f"⏱️ Retrieval: {t_retrieval:.2f}s | ⚡ Generation: {t_gen:.2f}s | 📊 Total: {total_time:.2f}s")
    print("=" * 70)
    print(answer)
    print("=" * 70 + "\n")
    return answer

# Test on Melioidosis vs Pulmonary Tuberculosis:
benchmark_query = (
    "What are the characteristic clinical symptoms, laboratory markers, and imaging findings "
    "of Melioidosis (Burkholderia pseudomallei) compared to pulmonary Tuberculosis?"
)

# Executes on GPU with 100% full context in ~20 seconds!
hybrid_resp = await run_diagnostic_query(benchmark_query, mode="hybrid", chunk_top_k=3)

---
## Step 5: Clinical Diagnostic Benchmark Suite 🩺
Four realistic medical queries specifically designed for the **Tropical Diseases Dataset**:

- **Scenario 1:** Complex Differential Diagnosis (Prolonged Fever & Hepatosplenomegaly)
- **Scenario 2:** Radiologic Findings & Diagnostic Modalities (Pulmonary TB vs Pneumonia)
- **Scenario 3:** Pharmacotherapy & Treatment Monitoring (MDR-TB & Severe Malaria)
- **Scenario 4:** Severe Disease Progression & Warning Signs (Dengue Shock Syndrome)

In [ ]:
clinical_cases = [
    {
        "title": "Scenario 1: Complex Differential Diagnosis",
        "query": (
            "A 28-year-old patient from Southeast Asia presents with 3 weeks of high remittent fever, "
            "marked hepatosplenomegaly, and pancytopenia. What tropical diseases should be considered, "
            "and what specific laboratory or microscopic tests confirm each condition?"
        ),
        "mode": "hybrid"
    },
    {
        "title": "Scenario 2: Radiologic & Imaging Distinctions",
        "query": (
            "What specific imaging patterns on chest computed tomography (CT) distinguish "
            "active pulmonary tuberculosis from opportunistic fungal infections or bacterial pneumonia?"
        ),
        "mode": "mix"
    },
    {
        "title": "Scenario 3: Pharmacotherapy & Toxicities",
        "query": (
            "What are the standard first-line treatment regimens and crucial organ toxicity monitoring "
            "parameters for severe falciparum malaria and multi-drug resistant tuberculosis?"
        ),
        "mode": "local"
    },
    {
        "title": "Scenario 4: Severe Disease Progression Signs",
        "query": (
            "What critical clinical and hematological warning signs indicate progression from "
            "uncomplicated dengue fever to severe dengue shock syndrome (DSS)?"
        ),
        "mode": "hybrid"
    }
]

# Run all 4 clinical benchmark scenarios (~3s each, ~12s total)
for idx, item in enumerate(clinical_cases, 1):
    print(f"\n{'#' * 70}")
    print(f"Case {idx}: {item['title']}")
    print(f"{'#' * 70}")
    await run_diagnostic_query(item["query"], mode=item["mode"], chunk_top_k=3)    await asyncio.sleep(2)  # Cooldown between intense clinical cases


---
## Step 6: Visual Graph Explorer & Clinical Reasoning Paths 🕸️
Inspects the actual structure of the Knowledge Graph:
1. **Ego-Network Subgraph:** Extracts and visualizes all entities connected to a central medical topic (e.g., *Tuberculosis* or *Dengue*).
2. **Shortest Reasoning Path:** Traces how two seemingly separate medical concepts are bridged through clinical evidence (e.g., *Symptom* $\rightarrow$ *Test* $\rightarrow$ *Pathogen* $\rightarrow$ *Drug*).

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def visualize_clinical_subgraph(G: nx.Graph, central_entity: str, max_neighbors: int = 15):
    """Visualize the 1-hop clinical neighborhood around a key disease or symptom."""
    if central_entity not in G:
        matches = [n for n in G.nodes if central_entity.lower() in n.lower()]
        if matches:
            print(f"Exact match '{central_entity}' not found. Using closest match: '{matches[0]}'")
            central_entity = matches[0]
        else:
            print(f"Entity '{central_entity}' not found in the Knowledge Graph.")
            return

    # Extract neighbors
    neighbors = list(G.neighbors(central_entity))[:max_neighbors]
    sub_nodes = [central_entity] + neighbors
    subgraph = G.subgraph(sub_nodes)

    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(subgraph, seed=42)

    # Color nodes: red for center, skyblue for neighbors
    node_colors = ['#e74c3c' if n == central_entity else '#3498db' for n in subgraph.nodes]

    nx.draw_networkx_nodes(subgraph, pos, node_color=node_colors, node_size=1600, alpha=0.9)
    nx.draw_networkx_edges(subgraph, pos, edge_color='#7f8c8d', width=1.8, alpha=0.6)
    nx.draw_networkx_labels(subgraph, pos, font_size=9, font_family='sans-serif', font_weight='bold')

    plt.title(f"Clinical Subgraph: 1-Hop Neighborhood of '{central_entity}'", fontsize=14, fontweight='bold', pad=15)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def find_clinical_reasoning_path(G: nx.Graph, source_entity: str, target_entity: str):
    """Find the shortest chain of clinical associations between two entities."""
    def match_node(query):
        if query in G: return query
        matches = [n for n in G.nodes if query.lower() in n.lower()]
        return matches[0] if matches else None

    src = match_node(source_entity)
    tgt = match_node(target_entity)

    if not src or not tgt:
        print(f"Could not resolve one of: '{source_entity}' ({src}) or '{target_entity}' ({tgt})")
        return

    try:
        path = nx.shortest_path(G, source=src, target=tgt)
        print(f"\n🔗 Clinical Reasoning Path between '{src}' and '{tgt}':")
        print("   " + "  -->  ".join([f"[{step}]" for step in path]))
        print(f"   (Path length: {len(path) - 1} diagnostic hops)")
    except nx.NetworkXNoPath:
        print(f"No direct path found between '{src}' and '{tgt}' in this graph component.")

# 1. Visualize ego-network around Tuberculosis (or Dengue)
visualize_clinical_subgraph(G, "Tuberculosis", max_neighbors=12)

# 2. Trace reasoning path from Symptom -> Drug
find_clinical_reasoning_path(G, "Fever", "Ceftriaxone")
find_clinical_reasoning_path(G, "Hemoptysis", "Isoniazid")

---
## Step 7: Interactive Diagnostic Console 💬
Type any custom clinical query below and press **Run** to query your Knowledge Graph!

**Available Modes:**
- `"hybrid"` — Best overall (graph traversal + vector retrieval)
- `"mix"` — Integrates graph nodes + raw case citations
- `"local"` — Fine-grained entity search
- `"global"` — Broad thematic patterns
- `"naive"` — Pure vector baseline (no graph)

In [ ]:
# Edit your question and preferred search mode here:
my_question = (
    "What are the common clinical presentations and diagnostic tests for Scrub Typhus (Orientia tsutsugamushi)?"
)
search_mode = "hybrid"  # "hybrid", "mix", "local", "global", or "naive"

# Run fast diagnostic query (~2-3 seconds)
response = await run_diagnostic_query(my_question, mode=search_mode, chunk_top_k=3)